# Gemini によるセマンティック マッチング

**モデルの説明:** このノートブックでは、Agent Platform で Gemini API を使用して製品説明をマッチングする方法を説明します。[Amazon-GoogleProducts データセット](https://dbs.uni-leipzig.de/file/Amazon-GoogleProducts.zip) には、Amazon または Google のウェブサイトからスクレイピングされた製品情報が含まれています。これには製品のタイトル、説明、価格、メーカーが含まれますが、これら 2 つのウェブサイトでは表現が異なっています。このノートブックでは、製品タイトルのみを使用したモデルの構築に焦点を当てます。考え方はシンプルで、製品タイトルが一致しているかどうかを Gemini API に問い合わせるプロンプトを作成します。製品に関する情報はこれらのタイトルに限定されていますが、テストセットでほぼ 100% の精度を達成できます。

**評価方法:** データセットに対するプロンプト設計の過学習（オーバーフィッティング）を避けるため、まずペアになった説明のデータセット サンプルを、60 件のサンプルを含む評価セット（evalset）と、同じく 60 件のサンプルからなるテストセット（testset）に分割します。60 件というサンプルの選択は、Agent Platform における Gemini API の現在のクォータ制限である 1 分あたり 60 リクエストに合わせています。その後、evalset を使用してプロンプトを考案し、testset でモデルの精度をレポートします。評価セットとテストセットはどちらもほぼ均衡（バランス）が取れています。


## セットアップ


In [ ]:
import random

import pandas as pd
from google import genai
from google.genai.types import GenerateContentConfig

pd.options.display.max_colwidth = 1000

## データセットの探索


Google および Amazon のウェブサイトからスクレイピングされた [製品情報のデータセット](https://dbs.uni-leipzig.de/file/Amazon-GoogleProducts.zip) を使用します。このデータセットは、ライプツィヒ大学による [セマンティック マッチングおよびエンティティ解決のベンチマーク](https://dbs.uni-leipzig.de/research/projects/object_matching/benchmark_datasets_for_entity_resolution) の一部です。このリポジトリに含まれている次の 3 つのテーブルで構成されています:

```python
../data/Amazon.csv.gz 
../data/GoogleProducts.csv.gz
../Amzon_GoogleProducts_perfectMapping.csv.gz

```

最初のテーブルには、製品のタイトル、説明、メーカーなど、Amazon に掲載されている製品情報が含まれています:


In [ ]:
amazon = pd.read_csv("../data/Amazon.csv.gz")
amazon.columns = [
    "idAmazon",
    "amazon_title",
    "amazon_description",
    "amazon_manufacturer",
    "amazon_price",
]
amazon.head()

2 番目のテーブルには同様の情報が含まれていますが、Google のウェブサイトからスクレイピングされた製品情報です:


In [ ]:
google = pd.read_csv("../data/GoogleProducts.csv.gz")
google.columns = [
    "idGoogleBase",
    "google_title",
    "google_description",
    "google_manufacturer",
    "google_price",
]
google.head()

最後のテーブルには、両方のウェブサイト上で同じ製品に対応する製品情報の一致（マッチング）が含まれていますが、2 つのウェブサイトで説明の表現が異なっている場合があります:


In [ ]:
matching = pd.read_csv("../data/Amzon_GoogleProducts_perfectMapping.csv.gz")
matching.head()

この生データから、評価セットとテストセットがあらかじめ生成されています:

```python
../data/product_matching_eval.csv
../data/product_matching_test.csv
```


eval セットを使用してプロンプトを設計し、test セットを使用してそのプロンプトに対する Gemini API のパフォーマンスを評価します。こうすることで、レポートするパフォーマンスは、これまで見たことのない製品説明のペアに対する実際のパフォーマンスに近くなります。


eval と test の分割を生成するために、下のセルの関数を使用しました。この関数は、`matching` DataFrame 内の一致する製品 ID のサンプル（サイズは `SAMPLE_SIZE` で制御）を取得し、`google` および `amazon` DataFrame に含まれる Google と Amazon の製品情報を結合します。次に、一致する Google と Amazon の説明のペアを抽出し、`description_1`（Google）、`description_2`（Amazon）、および現時点では一致するペアのみであるため値が `yes` に設定されたターゲット列 `match` を持つ一致説明テーブルを作成します。
一致しない製品の説明ペアを作成するために、最初の説明を固定したまま 2 番目の説明列を並べ替え（置換）、この一致しない説明の新しい DataFrame を一致する説明の DataFrame に連結します。シャッフルした後に結果のテーブルを等しいサイズの 2 つの DataFrame に分割し、eval 分割および test 分割としてディスクに保存します。

製品の説明として `title` 列のみを使用していることに注目してください。つまり、生データにはもっと多くの情報があります。それにもかかわらず、Gemini がテストセットで 100% に近いパフォーマンスを達成することがわかります。素晴らしい成果です！

**注:** データの別のサンプルで eval セットと test セットを再生成したい場合は、最後の行のコメントを解除してください。


In [ ]:
SAMPLE_SIZE = 60


def generate_test_and_eval_sets(sample_size=SAMPLE_SIZE):
    sample = matching.sample(sample_size)

    # 一致する ID の DataFrame に製品情報を結合
    matched_products = sample.merge(
        right=amazon, how="left", on="idAmazon"
    ).merge(right=google, how="left", on="idGoogleBase")

    google_descriptions = list(matched_products["google_title"])
    amazon_descriptions = list(matched_products["amazon_title"])

    # 一致する説明の DataFrame を作成
    matching_descriptions = pd.DataFrame(
        {
            "description_1": google_descriptions,
            "description_2": amazon_descriptions,
            "match": "yes",
        }
    )

    # 一致しない説明の DataFrame を作成
    amazon_descriptions_perm = [
        amazon_descriptions[i - 1] for i in range(len(amazon_descriptions))
    ]
    not_matching_descriptions = pd.DataFrame(
        {
            "description_1": google_descriptions,
            "description_2": amazon_descriptions_perm,
            "match": "no",
        }
    )

    full_dataset = pd.concat(
        [matching_descriptions, not_matching_descriptions], axis=0
    ).sample(len(matching_descriptions) * 2)

    evalset = full_dataset[:sample_size]
    testset = full_dataset[sample_size : 2 * sample_size]
    evalset.to_csv("../data/product_matching_eval.csv", index=None)
    testset.to_csv("../data/product_matching_test.csv", index=None)


# 異なるデータ サンプルを生成する場合はコメントを解除
# generate_test_and_eval_sets()


次のセルでは、事前生成された eval および test データセットを読み込みます。2 つの CSV ファイルには、製品説明のペアがそれぞれ 60 件含まれています。各ペアには、説明が同じ製品を表している場合は `match` の値として `yes` が、そうでない場合は `no` がラベル付けされています。`description_1` は Google の製品タイトル、`description_2` は Amazon の製品に由来します。


In [ ]:
evalset = pd.read_csv("../data/product_matching_eval.csv")
testset = pd.read_csv("../data/product_matching_test.csv")

このデータセットのエントリをいくつか簡単に確認してみましょう:


In [ ]:
evalset.head()

どちらの分割もほぼ均衡しています。両方の分割がほぼ均等であることを確認するために、各分割のクラス インスタンス数をカウントします。


In [ ]:
evalset.match.value_counts()

In [ ]:
testset.match.value_counts()

# モデルの実装


まず、Gen AI SDK を使用してクライアントをインスタンス化します。Google が開発した大規模言語モデル（LLM）である Gemini の `gemini-3.8-flash` バージョンを使用します。


In [ ]:
MODEL = "gemini-3.8-flash"

client = genai.Client(vertexai=True, location="global")

このクライアントを使用して、次のセルでシンプルな関数を実装します。この関数は、2 つの製品説明とパラメータ化されたプロンプトを入力として受け取り、Gemini モデルが製品説明が一致していると判断したかどうかに応じて `yes` または `no` を出力します。


### 演習

Gen AI SDK クライアントを使用して Gemini にクエリを実行するように、以下の関数を完成させてください。

**ヒント:** この関数がどのように使用されているかを確認するには、2 つ先のセルを参照してください。


In [ ]:
def are_products_matching(d1, d2, prompt):
    prompt = None  # TODO: パラメータ化されたプロンプトに d1 と d2 を代入
    answer = None  # TODO: プロンプトを使用して Gemini API を呼び出す
    return answer.text.strip()


次のセルでは、指定したプロンプトがこのユースケースで機能しているかどうかを評価セットで迅速にテストできます。


### 演習

下のセルで、2 つの製品説明が一致している場合は `yes`、そうでない場合は `no` と回答するよう Gemini に指示するプロンプトを記述してください。
実行時に異なる製品説明 `desc1` と `desc2` を切り替えられるように、プロンプトを `{desc1}` と `{desc2}` でパラメータ化してください。


In [ ]:
PROMPT = """
# TODO
"""

index = random.randint(0, len(evalset) - 1)

d1 = evalset.iloc[index]["description_1"]
d2 = evalset.iloc[index]["description_2"]
ground_truth = evalset.iloc[index]["match"]
prediction = are_products_matching(d1, d2, prompt=PROMPT)


print(
    f"""
Are the following two descriptions describing the same product?

Description 1: {d1}

Descriptions 2: {d2}

MODEL ANSWER: {prediction}
GROUND TRUTH: {ground_truth}
"""
)

# モデルの分析


それでは、テストセットにおけるモデルのパフォーマンスを分析しましょう。

大規模言語モデルは、プロンプトで丁寧にお願いした場合でも、"yes" や "no" 以外の出力を行うことがあります。これは、セーフティ フィルタがトリガーされたことや、モデルが単に質問を理解できなかったことが原因である可能性があります。そのため、まずモデルが回答に失敗したリクエストの割合を特定する必要があります。このケースでは約 6% であり、許容範囲内と言えるでしょう。さらなるプロンプト エンジニアリングやモデル チューニングによって、この数値を減らすことができる可能性があります。

考慮すべき 2 つ目の側面は、成功したリクエスト（つまり、出力が実際に `yes` または `no` であったリクエスト）に対するモデルのパフォーマンスです。
テストセットは均衡しているため、モデルの精度（accuracy）を計算でき、98% となります。これは、テストセット内で正解と異なる予測を受け取った例が 1 件しかなかったことを意味します。


## テストセットのスコアリング


評価をシンプルにするため、次のセルで `testset` に `prediction` 列を追加する関数を実装します。この列には、Gemini API から受信した予測結果が格納されます:


### 演習

作成したプロンプトを使用して、入力 DataFrame `dataset` に Gemini の予測結果を含む列 `prediction` を追加するように、以下の関数を完成させてください。


In [ ]:
def apply_prompt(prompt, dataset):
    scored_dataset = dataset.copy()
    # TODO: 以下の行を更新して、LLM の予測が DataFrame scored_dataset の predictions 列に保存されるようにしてください
    scored_dataset["predictions"] = None
    return scored_dataset


`evalset` 上で設計したシンプルなプロンプトを使用して、この関数を `testset` に適用してみましょう:


In [ ]:
scored_testset = apply_prompt(prompt=PROMPT, dataset=testset)

Gemini API へのリクエストは制限されており、1 分あたり 60 リクエストに制限されているため、必要に応じてオフラインで分析できるようにスコアリング結果をディスクに保存します。


In [ ]:
scored_testset.to_csv("scored_evalset.csv", index=False)

以下がモデルの予測結果です。ほとんどの例が正しく分類されていることがわかりますが、一部のリクエストが失敗し、予測が空になっているものもあります。これらのケースは個別に分析し、成功したリクエストに対してのみ精度を計算する必要があります:


In [ ]:
scored_testset.head(10)

### 失敗した予測


下のセルでは、失敗した予測、つまり `yes` または `no` 以外の予測の数を一覧表示します。このような動作にはいくつかの考えられる原因があります。Gemini は、すべての LLM と同様に、一連の単語から次に最も可能性の高い単語を予測するようにトレーニングされています。そのため、プロンプトで `yes` または `no` で答えるように明示的に指示した場合でも、特定の製品説明によって Gemini が混乱し、`yes` や `no` とは異なる結果になることがあります。もう 1 つの問題は、製品説明内の文言がセーフティ フィルタをトリガーし、Gemini の生の回答が標準の警告テキストに置き換えられることです。これらのセーフティ フィルタは、例えば健康や医療に関する問題、暴力などを強く連想させる製品説明内の単語や、他の多くの [セーフティ設定](https://developers.generativeai.google/guide/safety_setting) によってトリガーされる可能性があります。


### 演習

下のセルで、`scored_testset` に保存されている Gemini API の予測が失敗した場合は `True`、そうでない場合は `False` を含むブール値の Pandas Series `failed_predictions_mask` を作成するコードを記述してください。


In [ ]:
failed_predictions_mask = None  # TODO

failed_predictions_mask.value_counts()

In [ ]:
proportion_of_failed_predictions = failed_predictions_mask.sum() / len(
    failed_predictions_mask
)
print(
    "Proportion of failed requests:",
    round(proportion_of_failed_predictions, 3) * 100,
    "%",
)

次のセルでは、失敗したリクエストを調べます。一部の用語がセーフティ フィルタをトリガーした可能性がありますが、これにはさらなる調査が必要です:


In [ ]:
scored_testset[failed_predictions_mask]

## 成功した予測に対する精度（Accuracy）


成功したリクエストに対するモデルの精度を計算しましょう。まず、テストセットから失敗したすべてのリクエストを除外します:


In [ ]:
scored_testset_with_predictions = scored_testset[~failed_predictions_mask]

次に、正解の数を計算します:


### 演習

下のセルで、成功した予測に対する Gemini API の精度を計算してください。

**注:** モデルの精度は、正解の数を成功した予測の数で割った値です。


In [ ]:
# TODO: モデルの精度（accuracy）を計算


### 演習

下のセルで、検査のために `scored_testset_with_predictions` から誤った予測を抽出してください。


In [ ]:
# 検査用に誤った予測を抽出


Copyright 2023 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License